In [9]:
import os
import sys
import torch
import numpy as np
import cv2
from tqdm import tqdm

In [10]:
# source code
SRC_PATH = "/kaggle/input/datasets/priyanshusghosh/source/src"
sys.path.append(SRC_PATH)

# datasets
RAW_DIR = "/kaggle/input/datasets/iarunava/cell-images-for-detecting-malaria/cell_images/cell_images"
PROCESSED_DIR = "/kaggle/input/datasets/priyanshusghosh/processed"

# splits
SPLIT_DIR = "/kaggle/input/datasets/priyanshusghosh/splits"

# output
MODEL_DIR = "/kaggle/working/models"
os.makedirs(MODEL_DIR, exist_ok=True)

In [11]:
from data.loader import load_image_paths, read_image
from models.mobilenet import get_mobilenet
from train.train import train_model
from train.evaluate import evaluate
from utils.metrics import compute_metrics
from utils.latency import measure_latency

In [13]:
data = load_image_paths(RAW_DIR)

train_idx = np.load(os.path.join(SPLIT_DIR, "train_idx.npy"))
test_idx  = np.load(os.path.join(SPLIT_DIR, "test_idx.npy"))

train_data = [data[i] for i in train_idx]
test_data  = [data[i] for i in test_idx]

In [14]:
class SimpleDataset(torch.utils.data.Dataset):
    def __init__(self, data, processed=False):
        self.data = data
        self.processed = processed

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        path, label = self.data[idx]

        if self.processed:
            # map to processed path
            class_name = "Parasitized" if label == 0 else "Uninfected"
            filename = os.path.basename(path)
            path = os.path.join(PROCESSED_DIR, class_name, filename)
            img = cv2.imread(path)
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        else:
            img = read_image(path)

        img = cv2.resize(img, (224, 224))
        img = img / 255.0

        img = torch.tensor(img, dtype=torch.float32).permute(2, 0, 1)
        label = torch.tensor(label)

        return img, label

In [15]:
batch_size = 32

train_raw = torch.utils.data.DataLoader(
    SimpleDataset(train_data, processed=False),
    batch_size=batch_size,
    shuffle=True
)

test_raw = torch.utils.data.DataLoader(
    SimpleDataset(test_data, processed=False),
    batch_size=batch_size,
    shuffle=False
)

train_pre = torch.utils.data.DataLoader(
    SimpleDataset(train_data, processed=True),
    batch_size=batch_size,
    shuffle=True
)

test_pre = torch.utils.data.DataLoader(
    SimpleDataset(test_data, processed=True),
    batch_size=batch_size,
    shuffle=False
)

In [16]:
device = "cuda" if torch.cuda.is_available() else "cpu"

# RAW
model_raw = get_mobilenet()
model_raw, hist_raw = train_model(model_raw, train_raw, test_raw, epochs=5, device=device)

torch.save(model_raw.state_dict(), os.path.join(MODEL_DIR, "mobilenet_raw.pth"))

# PREPROCESSED
model_pre = get_mobilenet()
model_pre, hist_pre = train_model(model_pre, train_pre, test_pre, epochs=5, device=device)

torch.save(model_pre.state_dict(), os.path.join(MODEL_DIR, "mobilenet_pre.pth"))

/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=MobileNet_V3_Small_Weights.IMAGENET1K_V1`. You can also use `weights=MobileNet_V3_Small_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Downloading: "https://download.pytorch.org/models/mobilenet_v3_small-047dcff4.pth" to /root/.cache/torch/hub/checkpoints/mobilenet_v3_small-047dcff4.pth


100%|██████████| 9.83M/9.83M [00:00<00:00, 103MB/s]
100%|██████████| 689/689 [01:33<00:00,  7.40it/s]


In [22]:
def get_preds(model, loader):
    model.eval()
    preds, labels = [], []

    with torch.no_grad():
        for x, y in loader:
            x = x.to(device)
            out = model(x)
            p = torch.argmax(out, dim=1).cpu().numpy()

            preds.extend(p)
            labels.extend(y.numpy())

    return np.array(labels), np.array(preds)

In [18]:
# RAW
y_true_raw, y_pred_raw = get_preds(model_raw, test_raw)
metrics_raw = compute_metrics(y_true_raw, y_pred_raw)

# PRE
y_true_pre, y_pred_pre = get_preds(model_pre, test_pre)
metrics_pre = compute_metrics(y_true_pre, y_pred_pre)

print("RAW:", metrics_raw)
print("PRE:", metrics_pre)

RAW: {'accuracy': 0.9423076923076923, 'f1': 0.9445219818562456, 'precision': 0.9096102150537635, 'recall': 0.9822206095791002, 'confusion_matrix': array([[2487,  269],
       [  49, 2707]])}
PRE: {'accuracy': 0.9477503628447025, 'f1': 0.9495444989488437, 'precision': 0.9180216802168022, 'recall': 0.9833091436865021, 'confusion_matrix': array([[2514,  242],
       [  46, 2710]])}


In [19]:
lat_raw = measure_latency(model_raw, test_raw, device)
lat_pre = measure_latency(model_pre, test_pre, device)

print("Latency RAW:", lat_raw)
print("Latency PRE:", lat_pre)

Latency RAW: 3.3066076360350944
Latency PRE: 2.524930913493318


In [20]:
def get_size(path):
    return os.path.getsize(path) / (1024 * 1024)

size_raw = get_size(os.path.join(MODEL_DIR, "mobilenet_raw.pth"))
size_pre = get_size(os.path.join(MODEL_DIR, "mobilenet_pre.pth"))

In [21]:
import csv

csv_path = "/kaggle/working/metrics.csv"

rows = [
    ["MobileNet_Raw", metrics_raw["accuracy"], metrics_raw["f1"],
     metrics_raw["precision"], metrics_raw["recall"], lat_raw, size_raw],

    ["MobileNet_Pre", metrics_pre["accuracy"], metrics_pre["f1"],
     metrics_pre["precision"], metrics_pre["recall"], lat_pre, size_pre],
]

header = ["Model", "Accuracy", "F1", "Precision", "Recall", "Latency(ms)", "Size(MB)"]

if not os.path.exists(csv_path):
    with open(csv_path, "w", newline="") as f:
        writer = csv.writer(f)
        writer.writerow(header)
        writer.writerows(rows)
else:
    with open(csv_path, "a", newline="") as f:
        writer = csv.writer(f)
        writer.writerows(rows)

In [23]:
model = get_mobilenet()
model.load_state_dict(torch.load("/kaggle/working/models/mobilenet_pre.pth"))
model.eval()

/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=MobileNet_V3_Small_Weights.IMAGENET1K_V1`. You can also use `weights=MobileNet_V3_Small_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


MobileNetV3(
  (features): Sequential(
    (0): Conv2dNormActivation(
      (0): Conv2d(3, 16, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
      (1): BatchNorm2d(16, eps=0.001, momentum=0.01, affine=True, track_running_stats=True)
      (2): Hardswish()
    )
    (1): InvertedResidual(
      (block): Sequential(
        (0): Conv2dNormActivation(
          (0): Conv2d(16, 16, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), groups=16, bias=False)
          (1): BatchNorm2d(16, eps=0.001, momentum=0.01, affine=True, track_running_stats=True)
          (2): ReLU(inplace=True)
        )
        (1): SqueezeExcitation(
          (avgpool): AdaptiveAvgPool2d(output_size=1)
          (fc1): Conv2d(16, 8, kernel_size=(1, 1), stride=(1, 1))
          (fc2): Conv2d(8, 16, kernel_size=(1, 1), stride=(1, 1))
          (activation): ReLU()
          (scale_activation): Hardsigmoid()
        )
        (2): Conv2dNormActivation(
          (0): Conv2d(16, 16, kernel_size=(1, 1), 

In [24]:
from optimize.quantize import quantize_model

model_q = quantize_model(model)

In [29]:
def get_preds_cpu(model, loader):
    model.to("cpu")
    model.eval()

    preds, labels = [], []

    with torch.no_grad():
        for x, y in loader:
            x = x.to("cpu")   # force CPU
            out = model(x)
            p = torch.argmax(out, dim=1).numpy()

            preds.extend(p)
            labels.extend(y.numpy())

    return np.array(labels), np.array(preds)

In [30]:
y_true_q, y_pred_q = get_preds_cpu(model_q, test_pre)
metrics_q = compute_metrics(y_true_q, y_pred_q)

print(metrics_q)

{'accuracy': 0.9477503628447025, 'f1': 0.9495444989488437, 'precision': 0.9180216802168022, 'recall': 0.9833091436865021, 'confusion_matrix': array([[2514,  242],
       [  46, 2710]])}


In [31]:
lat_q = measure_latency(model_q, test_pre, device="cpu")  # IMPORTANT: CPU
print(lat_q)

7.288731494039859


In [32]:
torch.save(model_q.state_dict(), "/kaggle/working/models/mobilenet_quant.pth")

size_q = os.path.getsize("/kaggle/working/models/mobilenet_quant.pth") / (1024*1024)
print(size_q)

4.232420921325684


In [35]:
csv_path = "/kaggle/working/metrics.csv"

row = [
    "MobileNet_Quant",
    metrics_q["accuracy"],
    metrics_q["f1"],
    metrics_q["precision"],
    metrics_q["recall"],
    lat_q,
    size_q
]

with open(csv_path, "a", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(row)

In [36]:
import matplotlib.pyplot as plt
import seaborn as sns

def save_cm(cm, name):
    plt.figure(figsize=(4,4))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues")
    plt.title(name)
    plt.savefig(f"/kaggle/working/{name}.png")
    plt.close()

save_cm(metrics_raw["confusion_matrix"], "mobilenet_raw_cm")
save_cm(metrics_pre["confusion_matrix"], "mobilenet_pre_cm")
save_cm(metrics_q["confusion_matrix"], "mobilenet_quant_cm")

In [37]:
import matplotlib.pyplot as plt

# Loss curve
plt.plot(hist_pre["train_loss"], label="train")
plt.plot(hist_pre["val_loss"], label="val")
plt.legend()
plt.title("MobileNet Preprocessed - Loss")
plt.savefig("/kaggle/working/mobilenet_pre_loss.png")
plt.close()

# Accuracy curve
plt.plot(hist_pre["val_acc"], label="val_acc")
plt.legend()
plt.title("MobileNet Preprocessed - Accuracy")
plt.savefig("/kaggle/working/mobilenet_pre_acc.png")
plt.close()

In [38]:
import matplotlib.pyplot as plt

# Loss curve
plt.plot(hist_raw["train_loss"], label="train")
plt.plot(hist_raw["val_loss"], label="val")
plt.legend()
plt.title("MobileNet Raw - Loss")
plt.savefig("/kaggle/working/mobilenet_raw_loss.png")
plt.close()

# Accuracy curve
plt.plot(hist_raw["val_acc"], label="val_acc")
plt.legend()
plt.title("MobileNet Raw - Accuracy")
plt.savefig("/kaggle/working/mobilenet_raw_acc.png")
plt.close()